### Multi Linear Regression

We take housing price prediction dataset and perform data split feature transform and train validate the model

In [2]:
# import packages
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('housing_price_dataset.csv')
df.sample(4)

,SquareFeet,Bedrooms,Bathrooms,Neighborhood,YearBuilt,Price
21476,2672,3,2,Rural,1971,295221.452590
26105,2072,2,3,Urban,2003,141148.833001
41004,2141,2,1,Urban,1960,255330.588715
24939,2972,5,1,Rural,1950,288269.477679


#### Understand the dataset types and features

In [5]:
df.describe()

,SquareFeet,Bedrooms,Bathrooms,YearBuilt,Price
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,2006.374680,3.498700,1.995420,1985.404420,224827.325151
std,575.513241,1.116326,0.815851,20.719377,76141.842966
min,1000.000000,2.000000,1.000000,1950.000000,-36588.165397
25%,1513.000000,3.000000,1.000000,1967.000000,169955.860225
50%,2007.000000,3.000000,2.000000,1985.000000,225052.141166
75%,2506.000000,4.000000,3.000000,2003.000000,279373.630052
max,2999.000000,5.000000,3.000000,2021.000000,492195.259972


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   SquareFeet    50000 non-null  int64  
 1   Bedrooms      50000 non-null  int64  
 2   Bathrooms     50000 non-null  int64  
 3   Neighborhood  50000 non-null  str    
 4   YearBuilt     50000 non-null  int64  
 5   Price         50000 non-null  float64
dtypes: float64(1), int64(4), str(1)
memory usage: 2.3 MB


In [7]:
df.isnull().sum()

SquareFeet      0
Bedrooms        0
Bathrooms       0
Neighborhood    0
YearBuilt       0
Price           0
dtype: int64

In [8]:
df['Neighborhood'].value_counts()

Neighborhood
Suburb    16721
Rural     16676
Urban     16603
Name: count, dtype: int64

#### Extracting Features

In [9]:
# Drop the 'year' column
df.drop(columns=['YearBuilt'], inplace=True)

In [10]:
X = df.drop(columns=['Price'])
y = df['Price']

print(X)
print(y)

       SquareFeet  Bedrooms  Bathrooms Neighborhood
0            2126         4          1        Rural
1            2459         3          2        Rural
2            1860         2          1       Suburb
3            2294         2          1        Urban
4            2130         5          2       Suburb
...           ...       ...        ...          ...
49995        1282         5          3        Rural
49996        2854         2          2       Suburb
49997        2979         5          3       Suburb
49998        2596         5          2        Rural
49999        1572         5          3        Rural

[50000 rows x 4 columns]
0        215355.283618
1        195014.221626
2        306891.012076
3        206786.787153
4        272436.239065
             ...      
49995    100080.865895
49996    374507.656727
49997    384110.555590
49998    380512.685957
49999    221618.583218
Name: Price, Length: 50000, dtype: float64


#### Normalize Features

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_features = ['SquareFeet', 'Bedrooms', 'Bathrooms']
categorical_features = ['Neighborhood']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

#### Model Training

In [15]:
from sklearn.linear_model import SGDRegressor

sgd_model = SGDRegressor(loss="squared_error", max_iter=1000, learning_rate="invscaling", eta0=0.01)
sgd_model.fit(X_train, y_train)

print("coefficients:", sgd_model.coef_)
print("intercept:", sgd_model.intercept_)

coefficients: [56786.77339772  6737.50886539  3481.89675701 54284.46565496
 54171.88175015 56339.57472602]
intercept: [170271.2653757]


In [16]:
X_test

array([[-0.19463463,  1.34247591, -1.22207152,  1.        ,  0.        ,
         0.        ],
       [-1.74706239,  1.34247591,  1.22893434,  0.        ,  1.        ,
         0.        ],
       [ 0.44858852,  0.4456385 ,  1.22893434,  0.        ,  1.        ,
         0.        ],
       ...,
       [ 1.52816034, -0.4511989 ,  0.00343141,  0.        ,  0.        ,
         1.        ],
       [-0.57709164,  1.34247591,  0.00343141,  0.        ,  0.        ,
         1.        ],
       [ 0.38774308, -0.4511989 ,  0.00343141,  1.        ,  0.        ,
         0.        ]], shape=(10000, 6))

In [17]:
y_pred = sgd_model.predict(X_test)

In [19]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

rmse = root_mean_squared_error(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
r2 = r2_score(y_test, y_pred)

print(rmse)
print(mae)
print(r2)
print(rmse/mae)

49410.40012197719
39463.05121405197
0.574667686958175
1.252067404873731


In [20]:
# is the model underfitting or overfitting?
y_pred_train = sgd_model.predict(X_train)

r2_score_train = r2_score(y_train, y_pred_train)
r2_score_test = r2_score(y_test, y_pred)

print("train:", r2_score_train)
print("test:", r2_score_test)

train: 0.5684928147697247
test: 0.574667686958175
